# 📤 Notebook 05 — Dashboard Export
**Urban Taxi Demand Pattern Analysis**

This notebook aggregates the cleaned data and exports flat CSV files to `outputs/tableau/` for Tableau / Power BI dashboard consumption.

---

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
from pathlib import Path

from utils import DATA_PROCESSED, OUTPUT_TABLEAU, load_cleaned

OUTPUT_TABLEAU.mkdir(parents=True, exist_ok=True)

## 1. Load All Cleaned Data

In [ ]:
YEAR = 2024
MONTHS = list(range(1, 7))

dfs = []
for taxi_type in ['yellow', 'green']:
    for month in MONTHS:
        try:
            dfs.append(load_cleaned(taxi_type, YEAR, month))
        except FileNotFoundError:
            pass

df = pd.concat(dfs, ignore_index=True)
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
print(f"📊 Loaded {len(df):,} rows for export")

## 2. Hourly Demand Export
`demand_hourly.csv` — Trips per hour per day

In [ ]:
demand_hourly = (
    df.groupby([df['pickup_datetime'].dt.date.rename('date'),
                'pickup_hour'])
    .size()
    .reset_index(name='trip_count')
)

demand_hourly.to_csv(OUTPUT_TABLEAU / 'demand_hourly.csv', index=False)
print(f"✅ demand_hourly.csv — {len(demand_hourly):,} rows")
demand_hourly.head()

## 3. Daily Demand Export
`demand_daily.csv` — Trips per date

In [ ]:
demand_daily = (
    df.groupby(df['pickup_datetime'].dt.date.rename('date'))
    .agg(
        trip_count=('pickup_datetime', 'size'),
        avg_distance=('trip_distance', 'mean'),
        avg_fare=('fare_amount', 'mean'),
        avg_duration=('trip_duration_min', 'mean'),
        total_revenue=('total_amount', 'sum')
    )
    .reset_index()
)

demand_daily.to_csv(OUTPUT_TABLEAU / 'demand_daily.csv', index=False)
print(f"✅ demand_daily.csv — {len(demand_daily):,} rows")
demand_daily.head()

## 4. Zone Demand Export
`demand_zone.csv` — Trips per taxi zone

In [ ]:
demand_zone = (
    df.groupby('PULocationID')
    .agg(
        trip_count=('pickup_datetime', 'size'),
        avg_distance=('trip_distance', 'mean'),
        avg_fare=('fare_amount', 'mean'),
        avg_duration=('trip_duration_min', 'mean'),
        total_revenue=('total_amount', 'sum')
    )
    .reset_index()
    .sort_values('trip_count', ascending=False)
)

demand_zone.to_csv(OUTPUT_TABLEAU / 'demand_zone.csv', index=False)
print(f"✅ demand_zone.csv — {len(demand_zone):,} rows")
demand_zone.head(10)

## 5. Monthly Demand Export
`demand_monthly.csv` — Trips per month

In [ ]:
demand_monthly = (
    df.groupby('pickup_month')
    .agg(
        trip_count=('pickup_datetime', 'size'),
        avg_distance=('trip_distance', 'mean'),
        avg_fare=('fare_amount', 'mean'),
        avg_duration=('trip_duration_min', 'mean'),
        total_revenue=('total_amount', 'sum')
    )
    .reset_index()
)

month_names = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun',
               7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}
demand_monthly['month_name'] = demand_monthly['pickup_month'].map(month_names)

demand_monthly.to_csv(OUTPUT_TABLEAU / 'demand_monthly.csv', index=False)
print(f"✅ demand_monthly.csv — {len(demand_monthly):,} rows")
demand_monthly

## 6. Taxi Type Comparison Export

In [ ]:
if 'taxi_type' in df.columns:
    type_summary = (
        df.groupby('taxi_type')
        .agg(
            trip_count=('pickup_datetime', 'size'),
            avg_distance=('trip_distance', 'mean'),
            avg_fare=('fare_amount', 'mean'),
            avg_duration=('trip_duration_min', 'mean'),
            total_revenue=('total_amount', 'sum')
        )
        .reset_index()
    )

    type_summary.to_csv(OUTPUT_TABLEAU / 'taxi_type_summary.csv', index=False)
    print(f"✅ taxi_type_summary.csv")
    display(type_summary)

## 7. Export Validation
Quick sanity check that all CSV files are populated and null-free.

In [ ]:
print("\n📋 Export Validation")
print("=" * 60)

for csv_file in sorted(OUTPUT_TABLEAU.glob('*.csv')):
    tmp = pd.read_csv(csv_file)
    null_count = tmp.isnull().sum().sum()
    status = '✅' if null_count == 0 else f'⚠️ {null_count} nulls'
    print(f"  {status}  {csv_file.name:30s}  {len(tmp):>8,} rows  {len(tmp.columns)} cols")

print("\n🎉 All exports complete! Ready for Tableau import.")

---
**Pipeline complete!** Import the CSVs from `outputs/tableau/` into Tableau Public or Power BI to build the stakeholder dashboard.